In [6]:
# ==============================================================================
# The "Frankenstein" Model: BiGRU Dual Encoder + KL Divergence + Pairwise Ranking
# ==============================================================================
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q -o glove.6B.zip
!pip install -q optuna

import json
import os
import random
from contextlib import nullcontext

import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from torch.utils.data import DataLoader, Dataset, Sampler

# ------------------------------------------------------------------------------
# 1. Setup & Config
# ------------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

TRAIN_JSON = "/content/train.json"
TEST_JSON = "/content/test.json"
GLOVE_PATH = "/content/glove.6B.100d.txt"

MAX_WORDS = 10000
MAX_LEN = 150
EMBED_DIM = 100

TUNE_EPOCHS = 18
FINAL_EPOCHS = 36
PATIENCE = 5
N_TRIALS = 35

if not os.path.exists(TRAIN_JSON) or not os.path.exists(TEST_JSON):
    print("WARNING: Please upload train.json and test.json before running.")

# ------------------------------------------------------------------------------
# 2. Data Processing Helpers
# ------------------------------------------------------------------------------
def build_story(item):
    return " ".join([item.get("precontext", ""), item.get("sentence", ""), item.get("ending", "")])

def build_meaning(item):
    return item.get("judged_meaning", "")

def build_setup_key(item):
    return "|||".join([item.get("precontext", ""), item.get("sentence", ""), item.get("ending", "")])

def normalize_label_distribution(choices):
    """Converts the raw choices array into a 5-class probability distribution."""
    counts = np.zeros(5, dtype=np.float32)
    for c in choices:
        c = int(c)
        if 1 <= c <= 5:
            counts[c - 1] += 1.0
    total = counts.sum()
    if total == 0:
        return np.ones(5, dtype=np.float32) / 5.0
    return counts / total

def get_amp_autocast(device):
    if device.type != "cuda": return nullcontext()
    if hasattr(torch, "autocast"): return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True)
    return torch.cuda.amp.autocast(dtype=torch.float16, enabled=True)

def get_grad_scaler(device):
    amp_enabled = device.type == "cuda"
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"): return torch.amp.GradScaler(device.type, enabled=amp_enabled)
    return torch.cuda.amp.GradScaler(enabled=amp_enabled)

# ------------------------------------------------------------------------------
# 3. Load & Prepare Training Data
# ------------------------------------------------------------------------------
with open(TRAIN_JSON, encoding="utf8") as f:
    raw_train = json.load(f)

train_items = list(raw_train.values())
stories = [build_story(i) for i in train_items]
meanings = [build_meaning(i) for i in train_items]

# Labels and Distributions
labels = np.array([i["average"] for i in train_items], dtype=np.float32)
stdevs = np.array([i["stdev"] for i in train_items], dtype=np.float32)
label_dists = np.array([normalize_label_distribution(i.get("choices", [])) for i in train_items], dtype=np.float32)

# Groupings (To prevent leakage)
setup_keys = [build_setup_key(i) for i in train_items]
unique_setup = {k: idx for idx, k in enumerate(sorted(set(setup_keys)))}
setup_ids = np.array([unique_setup[k] for k in setup_keys], dtype=np.int64)

# Meta Features (The "Cheat Codes")
homonyms = [i.get("homonym", "<UNK>") for i in train_items]
unique_hom = {h: idx for idx, h in enumerate(sorted(set(homonyms)), start=1)}
hom_ids = np.array([unique_hom.get(h, 0) for h in homonyms], dtype=np.int64)

nons_count = np.array([sum(1 for v in i.get("nonsensical", []) if v) for i in train_items], dtype=np.float32)
ending_flag = np.array([1.0 if len(i.get("ending", "").strip()) > 0 else 0.0 for i in train_items], dtype=np.float32)
meta_feats = np.stack([np.clip(nons_count / 5.0, 0.0, 1.0), ending_flag], axis=1).astype(np.float32)

# Tokenization & Padding
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(stories + meanings)
X_story = pad_sequences(tokenizer.texts_to_sequences(stories), maxlen=MAX_LEN, padding="post")
X_mean = pad_sequences(tokenizer.texts_to_sequences(meanings), maxlen=MAX_LEN, padding="post")

# Group Shuffle Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(gss.split(X_story, labels, groups=setup_ids))

Xs_train, Xs_val = X_story[train_idx], X_story[val_idx]
Xm_train, Xm_val = X_mean[train_idx], X_mean[val_idx]
y_train, y_val = labels[train_idx], labels[val_idx]
dist_train, dist_val = label_dists[train_idx], label_dists[val_idx]
s_train, s_val = stdevs[train_idx], stdevs[val_idx]
setup_train, setup_val = setup_ids[train_idx], setup_ids[val_idx]
hom_train, hom_val = hom_ids[train_idx], hom_ids[val_idx]
meta_train, meta_val = meta_feats[train_idx], meta_feats[val_idx]

# GloVe Embeddings
embeddings_index = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        vals = line.split()
        embeddings_index[vals[0]] = np.asarray(vals[1:], dtype="float32")

word_index = tokenizer.word_index
num_words = min(MAX_WORDS, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, EMBED_DIM), dtype=np.float32)
for word, idx in word_index.items():
    if idx < num_words and word in embeddings_index:
        embedding_matrix[idx] = embeddings_index[word]

# ------------------------------------------------------------------------------
# 4. PyTorch Dataset & Models
# ------------------------------------------------------------------------------
class AmbiStoryDataset(Dataset):
    def __init__(self, X_story, X_mean, labels, dists, stdevs, setup_ids, hom_ids, meta_feats):
        self.X_story = torch.tensor(X_story, dtype=torch.long)
        self.X_mean = torch.tensor(X_mean, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.dists = torch.tensor(dists, dtype=torch.float32)
        self.stdevs = torch.tensor(stdevs, dtype=torch.float32)
        self.setup_ids = torch.tensor(setup_ids, dtype=torch.long)
        self.hom_ids = torch.tensor(hom_ids, dtype=torch.long)
        self.meta = torch.tensor(meta_feats, dtype=torch.float32)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return (self.X_story[idx], self.X_mean[idx], self.labels[idx], self.dists[idx],
                self.stdevs[idx], self.setup_ids[idx], self.hom_ids[idx], self.meta[idx])

class SetupBatchSampler(Sampler):
    """Ensures pairs from the same setup are in the same batch for ranking loss."""
    def __init__(self, setup_ids, setups_per_batch=8, shuffle=True):
        self.shuffle = shuffle
        self.setups_per_batch = setups_per_batch
        self.indices_by_setup = {}
        for idx, sid in enumerate(setup_ids):
            self.indices_by_setup.setdefault(int(sid), []).append(idx)
        self.setup_keys = list(self.indices_by_setup.keys())

    def __iter__(self):
        keys = self.setup_keys.copy()
        if self.shuffle: random.shuffle(keys)
        for start in range(0, len(keys), self.setups_per_batch):
            selected = keys[start:start + self.setups_per_batch]
            batch = []
            for sid in selected:
                inds = self.indices_by_setup[sid]
                if self.shuffle: random.shuffle(inds)
                batch.extend(inds)
            if batch: yield batch

    def __len__(self):
        return (len(self.setup_keys) + self.setups_per_batch - 1) // self.setups_per_batch

class BiGRUEncoder(nn.Module):
    def __init__(self, num_words, embed_dim, embedding_matrix, rnn_units, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(num_words, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.gru = nn.GRU(input_size=embed_dim, hidden_size=rnn_units, num_layers=num_layers,
                          batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)
        out, _ = self.gru(x)
        mean_pool = out.mean(dim=1)
        max_pool = out.max(dim=1).values
        return torch.cat([mean_pool, max_pool], dim=1)

class DistributionalDualEncoder(nn.Module):
    def __init__(self, num_words, embed_dim, embedding_matrix, rnn_units, num_layers, dropout, dense_units, hom_vocab_size, hom_emb_dim, meta_dim):
        super().__init__()
        encoder_args = (num_words, embed_dim, embedding_matrix, rnn_units, num_layers, dropout)
        self.story_encoder = BiGRUEncoder(*encoder_args)
        self.meaning_encoder = BiGRUEncoder(*encoder_args)
        self.hom_emb = nn.Embedding(hom_vocab_size + 1, hom_emb_dim)

        base_dim = rnn_units * 4
        combined_dim = (base_dim * 4) + 1 + hom_emb_dim + meta_dim

        # CHANGED: Now outputs 5 classes instead of 1 for KL Divergence
        self.head = nn.Sequential(
            nn.Linear(combined_dim, dense_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dense_units, 5),
        )

    def forward(self, story, meaning, hom_ids, meta):
        sv = self.story_encoder(story)
        mv = self.meaning_encoder(meaning)
        abs_diff = torch.abs(sv - mv)
        prod = sv * mv
        cosine = F.cosine_similarity(sv, mv, dim=1, eps=1e-8).unsqueeze(1)
        hom_vec = self.hom_emb(hom_ids)

        combined = torch.cat([sv, mv, abs_diff, prod, cosine, hom_vec, meta], dim=1)
        return self.head(combined) # Returns 5 logits

# ------------------------------------------------------------------------------
# 5. Losses & Training Loop
# ------------------------------------------------------------------------------
def get_expected_scores(logits):
    """Converts 5-class logits into a continuous 1-5 expected score."""
    probs = torch.softmax(logits, dim=-1)
    weights = torch.arange(1, 6, device=logits.device, dtype=torch.float32)
    return (probs * weights).sum(dim=-1)

def pairwise_setup_ranking_loss(preds, targets, setup_ids, margin=0.2):
    total_loss = preds.new_tensor(0.0)
    groups = 0
    unique_setups = torch.unique(setup_ids)
    for sid in unique_setups:
        mask = setup_ids == sid
        p, y = preds[mask], targets[mask]
        if p.numel() < 2: continue
        y_diff = y.unsqueeze(1) - y.unsqueeze(0)
        p_diff = p.unsqueeze(1) - p.unsqueeze(0)
        valid = y_diff.abs() > 1e-6
        if valid.sum() == 0: continue
        loss_mat = torch.relu(margin - y_diff.sign() * p_diff)
        total_loss = total_loss + (loss_mat * valid).sum() / valid.sum()
        groups += 1
    if groups == 0: return preds.new_tensor(0.0)
    return total_loss / groups

def create_loaders(batch_size_setups):
    workers = 2 if device.type == "cuda" else 0
    train_ds = AmbiStoryDataset(Xs_train, Xm_train, y_train, dist_train, s_train, setup_train, hom_train, meta_train)
    val_ds = AmbiStoryDataset(Xs_val, Xm_val, y_val, dist_val, s_val, setup_val, hom_val, meta_val)
    train_sampler = SetupBatchSampler(setup_train, setups_per_batch=batch_size_setups, shuffle=True)
    train_loader = DataLoader(train_ds, batch_sampler=train_sampler, pin_memory=(device.type == "cuda"), num_workers=workers)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, pin_memory=(device.type == "cuda"), num_workers=workers)
    return train_loader, val_loader

def train_one_epoch(model, loader, optimizer, scaler, huber_loss_fn, rank_lambda, rank_margin, huber_lambda):
    model.train()
    total_loss = 0.0

    for story, meaning, label, dist, _stdev, sid, hom, meta in loader:
        story, meaning, label = story.to(device), meaning.to(device), label.to(device)
        dist, sid, hom, meta = dist.to(device), sid.to(device), hom.to(device), meta.to(device)

        optimizer.zero_grad(set_to_none=True)

        with get_amp_autocast(device):
            logits = model(story, meaning, hom, meta)
            expected_scores = get_expected_scores(logits)

            # The Ultimate Tri-Loss Setup
            loss_kl = F.kl_div(F.log_softmax(logits, dim=-1), dist, reduction='batchmean')
            loss_rank = pairwise_setup_ranking_loss(expected_scores, label, sid, margin=rank_margin)
            loss_huber = huber_loss_fn(expected_scores, label)

            loss = loss_kl + (rank_lambda * loss_rank) + (huber_lambda * loss_huber)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / max(len(loader), 1)

@torch.no_grad()
def predict(model, Xs, Xm, hom_ids_arr, meta_arr):
    model.eval()
    dummy_y = np.zeros(len(Xs), dtype=np.float32)
    dummy_d = np.zeros((len(Xs), 5), dtype=np.float32)
    dummy_s = np.ones(len(Xs), dtype=np.float32)
    dummy_sid = np.arange(len(Xs), dtype=np.int64)

    ds = AmbiStoryDataset(Xs, Xm, dummy_y, dummy_d, dummy_s, dummy_sid, hom_ids_arr, meta_arr)
    loader = DataLoader(ds, batch_size=64, shuffle=False, pin_memory=(device.type == "cuda"))

    all_preds = []
    for story, meaning, _label, _dist, _stdev, _sid, hom, meta in loader:
        story, meaning, hom, meta = story.to(device), meaning.to(device), hom.to(device), meta.to(device)
        logits = model(story, meaning, hom, meta)
        expected_scores = get_expected_scores(logits)
        all_preds.append(expected_scores.cpu().numpy())

    return np.clip(np.concatenate(all_preds), 1, 5)

def objective_score(y_true, y_pred, s_true):
    sp = spearmanr(y_true, y_pred).correlation
    if np.isnan(sp): sp = -1.0
    acc = np.mean(np.abs(y_pred - y_true) <= np.maximum(s_true, 1.0))
    mae = np.mean(np.abs(y_pred - y_true))
    # Direct Optimization against the SemEval leaderboard logic
    return float(sp + 0.25 * acc - 0.05 * mae), float(sp), float(acc), float(mae)

def run_training(params, epochs, patience, trial=None):
    model = DistributionalDualEncoder(
        num_words=num_words, embed_dim=EMBED_DIM, embedding_matrix=embedding_matrix,
        rnn_units=params["rnn_units"], num_layers=params["num_layers"], dropout=params["dropout"],
        dense_units=params["dense_units"], hom_vocab_size=len(unique_hom), hom_emb_dim=params["hom_emb_dim"], meta_dim=meta_train.shape[1],
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
    scaler = get_grad_scaler(device)
    huber_loss_fn = nn.HuberLoss(reduction="mean", delta=1.0)

    train_loader, val_loader = create_loaders(params["setup_batch"])

    best_score = -1e9
    best_weights = None
    patience_ctr = 0

    for epoch in range(1, epochs + 1):
        _ = train_one_epoch(
            model, train_loader, optimizer, scaler, huber_loss_fn,
            rank_lambda=params["rank_lambda"], rank_margin=params["rank_margin"], huber_lambda=params["huber_lambda"]
        )
        val_preds = predict(model, Xs_val, Xm_val, hom_val, meta_val)
        score, sp, acc, mae = objective_score(y_val, val_preds, s_val)
        scheduler.step(score)

        if trial is not None:
            trial.report(score, step=epoch)
            if trial.should_prune(): raise optuna.TrialPruned()

        if score > best_score:
            best_score = score
            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience: break

    if best_weights is not None: model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})

    final_preds = predict(model, Xs_val, Xm_val, hom_val, meta_val)
    score, sp, acc, mae = objective_score(y_val, final_preds, s_val)
    return model, score, sp, acc, mae

# ------------------------------------------------------------------------------
# 6. Optuna Search & Execution
# ------------------------------------------------------------------------------
def objective(trial):
    params = {
        "rnn_units": trial.suggest_categorical("rnn_units", [64, 96, 128]),
        "num_layers": trial.suggest_categorical("num_layers", [1, 2]),
        "dropout": trial.suggest_float("dropout", 0.15, 0.45),
        "dense_units": trial.suggest_categorical("dense_units", [96, 128, 192, 256]),
        "hom_emb_dim": trial.suggest_categorical("hom_emb_dim", [8, 16, 24]),
        "lr": trial.suggest_float("lr", 1e-4, 2e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 5e-4, log=True),
        "setup_batch": trial.suggest_categorical("setup_batch", [6, 8, 10, 12]),
        "rank_lambda": trial.suggest_float("rank_lambda", 0.2, 0.9),
        "rank_margin": trial.suggest_float("rank_margin", 0.1, 0.6),
        "huber_lambda": trial.suggest_float("huber_lambda", 0.1, 1.0), # Added tuning for the auxiliary Huber loss
    }
    _, score, _, _, _ = run_training(params, epochs=TUNE_EPOCHS, patience=PATIENCE, trial=trial)
    return score

def make_test_meta_and_hom(test_items):
    test_hom = np.array([unique_hom.get(i.get("homonym", "<UNK>"), 0) for i in test_items], dtype=np.int64)
    test_nons = np.array([sum(1 for v in i.get("nonsensical", []) if v) for i in test_items], dtype=np.float32)
    test_end = np.array([1.0 if len(i.get("ending", "").strip()) > 0 else 0.0 for i in test_items], dtype=np.float32)
    test_meta = np.stack([np.clip(test_nons / 5.0, 0.0, 1.0), test_end], axis=1).astype(np.float32)
    return test_hom, test_meta

if __name__ == "__main__":
    if os.path.exists(TRAIN_JSON) and os.path.exists(TEST_JSON):
        sampler = optuna.samplers.TPESampler(seed=SEED)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=5)
        study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

        print("\nBest trial:")
        print(f"  Value (composite): {study.best_value:.5f}")
        for key, value in study.best_params.items():
            print(f"  {key}: {value}")

        best_model, _, val_sp, val_acc, val_mae = run_training(study.best_params, epochs=FINAL_EPOCHS, patience=7, trial=None)

        print("\nValidation with best params:")
        print("Spearman r           :", val_sp)
        print("Accuracy within std  :", val_acc)
        print("MAE                  :", val_mae)

        with open(TEST_JSON, encoding="utf8") as f:
            raw_test = json.load(f)

        test_items = list(raw_test.values())
        test_story = pad_sequences(tokenizer.texts_to_sequences([build_story(i) for i in test_items]), maxlen=MAX_LEN, padding="post")
        test_mean = pad_sequences(tokenizer.texts_to_sequences([build_meaning(i) for i in test_items]), maxlen=MAX_LEN, padding="post")
        test_hom, test_meta = make_test_meta_and_hom(test_items)

        test_preds = predict(best_model, test_story, test_mean, test_hom, test_meta)

        # Save Predictions to JSONL for SemEval Official Submission format
        submission_lines = []
        for i, pred in enumerate(test_preds):
            submission_lines.append(json.dumps({"id": str(i), "prediction": max(1, min(5, int(round(float(pred)))))}))

        with open("/content/submission.jsonl", "w") as f:
            f.write("\n".join(submission_lines) + "\n")

        print("\nSaved Official Submission Format:")
        print("  /content/submission.jsonl")

    else:
        print("Waiting for datasets to be uploaded...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 13.2 MB/s eta 0:00:00
Using device: cuda


[I 2026-03-06 15:24:04,719] A new study created in memory with name: no-name-911a415a-a0d9-4a72-8a49-f29765e79d46


  0%|          | 0/35 [00:00<?, ?it/s]

[I 2026-03-06 15:24:51,331] Trial 0 finished with value: 0.5345554597065585 and parameters: {'rnn_units': 96, 'num_layers': 1, 'dropout': 0.1967983561008608, 'dense_units': 128, 'hom_emb_dim': 16, 'lr': 0.00018891200276189413, 'weight_decay': 3.0955664602423687e-06, 'setup_batch': 10, 'rank_lambda': 0.40386039813862934, 'rank_margin': 0.40592644736118977, 'huber_lambda': 0.22554447458683766}. Best is trial 0 with value: 0.5345554597065585.
[I 2026-03-06 15:25:44,433] Trial 1 finished with value: 0.7086771607015698 and parameters: {'rnn_units': 128, 'num_layers': 1, 'dropout': 0.3042703315240835, 'dense_units': 192, 'hom_emb_dim': 24, 'lr': 0.0011265466963346032, 'weight_decay': 6.639623079859462e-06, 'setup_batch': 8, 'rank_lambda': 0.5466238370778891, 'rank_margin': 0.1171942605576092, 'huber_lambda': 0.9183883618709039}. Best is trial 1 with value: 0.7086771607015698.
[I 2026-03-06 15:26:31,106] Trial 2 finished with value: 0.5162126816307154 and parameters: {'rnn_units': 96, 'num_la

In [ ]:
# ============================================================
# FIGURE 2/3/4 EXPORTS (KL model)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

if "best_model" not in globals():
    raise RuntimeError("Run the training cell first so best_model and study are available.")

# Validation predictions from best model
val_preds = predict(best_model, Xs_val, Xm_val, hom_val, meta_val)
stdevs = np.maximum(s_val, 1.0)
residuals = val_preds - y_val
within = np.abs(residuals) <= stdevs
spear = spearmanr(y_val, val_preds).correlation
mae = np.mean(np.abs(residuals))

# FIGURE 2: Hyperparameter tuning scatter
if "study" in globals() and getattr(study, "trials", None):
    completed = [t for t in study.trials if t.value is not None]
    if len(completed) > 0:
        fig, axes = plt.subplots(2, 3, figsize=(16, 9))
        fig.suptitle("Figure 2 - KL Model Hyperparameter Tuning", fontsize=16, fontweight="bold")

        params = [
            ("rnn_units", "RNN Units"),
            ("num_layers", "Num Layers"),
            ("dropout", "Dropout"),
            ("dense_units", "Dense Units"),
            ("lr", "Learning Rate"),
            ("rank_lambda", "Rank Lambda"),
        ]

        for ax, (param, title) in zip(axes.flat, params):
            xs = []
            ys = []
            for t in completed:
                if param in t.params:
                    xs.append(t.params[param])
                    ys.append(t.value)
            if len(xs) > 0:
                ax.scatter(xs, ys, alpha=0.8, s=70, edgecolors="white")
            ax.set_xlabel(title)
            ax.set_ylabel("Composite Score")
            ax.set_title(f"{title} vs Objective")

        plt.tight_layout()
        plt.savefig("/content/trial3_model8_figure2.png", dpi=150, bbox_inches="tight")
        plt.show()

        # FIGURE 3: Top 10 trials
        ranked = sorted(completed, key=lambda t: t.value, reverse=True)[:10]
        labels = [f"Trial {t.number}" for t in ranked]
        scores = [t.value for t in ranked]

        fig, ax = plt.subplots(figsize=(12, 5))
        bars = ax.barh(labels[::-1], scores[::-1], color="steelblue", alpha=0.88)
        if len(bars) > 0:
            bars[-1].set_color("#4CAF50")
            bars[0].set_color("#F44336")

        for bar, score in zip(bars, scores[::-1]):
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2, f"{score:.4f}", va="center")

        ax.set_xlabel("Composite Objective (higher = better)")
        ax.set_title("Figure 3 - Top 10 Optuna Trials (KL Model)")
        plt.tight_layout()
        plt.savefig("/content/trial3_model8_figure3.png", dpi=150, bbox_inches="tight")
        plt.show()

# FIGURE 4: Summary card
fig = plt.figure(figsize=(10, 4))
fig.patch.set_facecolor("#1E1E2E")
ax = fig.add_subplot(111)
ax.set_facecolor("#1E1E2E")
ax.axis("off")

metrics = [
    ("Spearman r", f"{float(spear):.4f}", "#4CAF50"),
    ("Acc within Std", f"{float(np.mean(within)):.4f}", "#2196F3"),
    ("Mean Residual", f"{float(np.mean(residuals)):.4f}", "#FF9800"),
    ("MAE", f"{float(mae):.4f}", "#9C27B0"),
]

for i, (label, value, color) in enumerate(metrics):
    x = 0.12 + i * 0.24
    ax.text(x, 0.72, value, transform=ax.transAxes, fontsize=26, fontweight="bold", color=color, ha="center")
    ax.text(x, 0.45, label, transform=ax.transAxes, fontsize=11, color="white", ha="center", alpha=0.85)
    if i < len(metrics) - 1:
        ax.axvline(x + 0.12, color="white", alpha=0.15, lw=1)

ax.text(0.5, 0.92, "Figure 4 - KL Model Summary", transform=ax.transAxes, fontsize=13, color="white", ha="center", alpha=0.7)
plt.tight_layout()
plt.savefig("/content/trial3_model8_figure4.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print("\nSaved standardized Trial 3 (Model 8) figures:")
print("  /content/trial3_model8_figure2.png")
print("  /content/trial3_model8_figure3.png")
print("  /content/trial3_model8_figure4.png")